# Data Understanding
<hr>

### Imports

In [4]:
import os

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
# load in the data from data folder
df = pd.read_csv('../data/allstate_claims_data.csv')

In [6]:
# CHECK FILE SIZE
actual_size = os.path.getsize('../data/allstate_claims_data.csv')
expected_size = 70025339

print("File size:", actual_size)

if actual_size != expected_size:
    print(f"File size does not match expected: expected {expected_size}, got {actual_size}", "\n")
else:
    print("File size matches", "\n")


# CHECK DF ROWS AND COLUMNS
actual_shape = df.shape
expected_shape = (188318, 132)

print("Rows:", actual_shape[0])
print("Columns:", actual_shape[1])

if actual_shape != expected_shape:
    print(f"df does not match expected rows and columns: expected {expected_shape}, got {actual_shape}")
else:
    print("df matches")

File size: 70025339
File size matches 

Rows: 188318
Columns: 132
df matches


In [7]:
# CHECK HEADERS MATCH
df.head()

,id,cat1,cat2,cat3,cat4,cat5,cat6,cat7,cat8,cat9,...,cont6,cont7,cont8,cont9,cont10,cont11,cont12,cont13,cont14,loss
0,1,A,B,A,B,A,A,A,A,B,...,0.718367,0.335060,0.30260,0.67135,0.83510,0.569745,0.594646,0.822493,0.714843,2213.18
1,2,A,B,A,A,A,A,A,A,B,...,0.438917,0.436585,0.60087,0.35127,0.43919,0.338312,0.366307,0.611431,0.304496,1283.60
2,5,A,B,A,A,B,A,A,A,B,...,0.289648,0.315545,0.27320,0.26076,0.32446,0.381398,0.373424,0.195709,0.774425,3005.09
3,10,B,B,A,B,A,A,A,A,B,...,0.440945,0.391128,0.31796,0.32128,0.44467,0.327915,0.321570,0.605077,0.602642,939.85
4,11,A,B,A,B,A,A,A,A,B,...,0.178193,0.247408,0.24564,0.22089,0.21230,0.204687,0.202213,0.246011,0.432606,2763.85


In [8]:
## Task 2 — Column Roles and Basic Integrity
## Assignee: Leng Lim

In [9]:
# CONFIRM COLUMN ROLES
cat_cols = [f"cat{i}" for i in range(1, 117)]
cont_cols = [f"cont{i}" for i in range(1, 15)]

print("Identifier present:", "id" in df.columns)
print("Categorical predictors found:", sum(col in df.columns for col in cat_cols))
print("Continuous predictors found:", sum(col in df.columns for col in cont_cols))
print("Regression target present:", "loss" in df.columns)

Identifier present: True
Categorical predictors found: 116
Continuous predictors found: 14
Regression target present: True


In [10]:

print("ID present:", "id" in df.columns)
print("Missing IDs:", df["id"].isna().sum())
print("Unique IDs:", df["id"].nunique())
print("Total rows:", len(df))
print("All IDs unique:", df["id"].is_unique)

ID present: True
Missing IDs: 0
Unique IDs: 188318
Total rows: 188318
All IDs unique: True


In [11]:
df.isna()
df.isna().sum()
df.isna().sum()
print("Total missing cells:", df.isna().sum().sum())

Total missing cells: 0


In [12]:
df.duplicated()
df.duplicated().sum()
print("Exact duplicate rows:", df.duplicated().sum())

Exact duplicate rows: 0


In [13]:
all_numeric = all(
    pd.api.types.is_numeric_dtype(df[col])
    for col in cont_cols
)

all_in_range = (
    (df[cont_cols] >= 0) &
    (df[cont_cols] <= 1)
).all().all()

print("All continuous columns numeric:", all_numeric)
print("All continuous values within 0-1:", all_in_range)


All continuous columns numeric: True
All continuous values within 0-1: True


## Task 3 — Field Validation and Source Profile


In [14]:
# PART 1 — VALIDATE THE loss TARGET
loss_series = df['loss']
loss_dtype = loss_series.dtype
loss_unique_count = loss_series.nunique(dropna=False)
loss_missing_count = loss_series.isna().sum()
loss_min = loss_series.min()
loss_max = loss_series.max()
loss_non_finite_count = np.isinf(pd.to_numeric(loss_series, errors='coerce')).sum()
loss_non_positive_count = (pd.to_numeric(loss_series, errors='coerce') <= 0).sum()

loss_numeric = pd.api.types.is_numeric_dtype(loss_series)
loss_no_missing = loss_missing_count == 0
loss_no_infinite = loss_non_finite_count == 0
loss_positive = (pd.to_numeric(loss_series, errors='coerce') > 0).all()

print('Loss dtype:', loss_dtype)
print('Loss unique count:', loss_unique_count)
print('Loss missing count:', loss_missing_count)
print('Loss min:', loss_min)
print('Loss max:', loss_max)
print('Loss non-finite count:', loss_non_finite_count)
print('Loss values <= 0:', loss_non_positive_count)
print('Loss numeric:', 'PASS' if loss_numeric else 'FAIL')
print('Loss no missing values:', 'PASS' if loss_no_missing else 'FAIL')
print('Loss no infinite values:', 'PASS' if loss_no_infinite else 'FAIL')
print('Loss strictly positive:', 'PASS' if loss_positive else 'FAIL')

# PART 2 — VALIDATE CATEGORICAL COLUMNS
categorical_profile = pd.DataFrame({
    'column_name': cat_cols,
    'observed_dtype': [df[col].dtype for col in cat_cols],
    'unique_count': [df[col].nunique(dropna=True) for col in cat_cols],
    'missing_count': [df[col].isna().sum() for col in cat_cols],
    'missing_percentage': [df[col].isna().mean() * 100 for col in cat_cols],
})

categorical_profile.head()
print('Categorical profile rows:', len(categorical_profile))
print('Categorical profile sample:')
categorical_profile.head()

# PART 3 — CREATE THE COMPLETE SOURCE PROFILE
roles = {}
for col in df.columns:
    if col == 'id':
        roles[col] = 'identifier'
    elif col in cat_cols:
        roles[col] = 'categorical_predictor'
    elif col in cont_cols:
        roles[col] = 'continuous_predictor'
    elif col == 'loss':
        roles[col] = 'regression_target'
    else:
        roles[col] = 'unknown'

source_profile = pd.DataFrame({
    'column_name': df.columns,
    'role': [roles[col] for col in df.columns],
    'observed_dtype': [df[col].dtype for col in df.columns],
    'unique_count': [df[col].nunique(dropna=True) for col in df.columns],
    'missing_count': [df[col].isna().sum() for col in df.columns],
    'missing_percentage': [df[col].isna().mean() * 100 for col in df.columns],
    'observed_min': np.nan,
    'observed_max': np.nan,
    'expected_scale': [
        'N/A' if col == 'id' else
        'nominal categorical' if col in cat_cols else
        '[0, 1]' if col in cont_cols else
        'positive continuous' if col == 'loss' else
        'unknown'
        for col in df.columns
    ],
})

for col in df.columns:
    if col in cat_cols:
        source_profile.loc[source_profile['column_name'] == col, 'observed_min'] = np.nan
        source_profile.loc[source_profile['column_name'] == col, 'observed_max'] = np.nan
    elif pd.api.types.is_numeric_dtype(df[col]):
        source_profile.loc[source_profile['column_name'] == col, 'observed_min'] = df[col].min()
        source_profile.loc[source_profile['column_name'] == col, 'observed_max'] = df[col].max()
    else:
        source_profile.loc[source_profile['column_name'] == col, 'observed_min'] = np.nan
        source_profile.loc[source_profile['column_name'] == col, 'observed_max'] = np.nan

source_profile.head()
print('Source profile rows:', len(source_profile))
source_profile.head()

# PART 4 — VERIFY THE PROFILE
profile_rows_ok = len(source_profile) == 132
profile_unique_ok = source_profile['column_name'].is_unique
profile_cat_count_ok = (source_profile['role'] == 'categorical_predictor').sum() == 116
profile_cont_count_ok = (source_profile['role'] == 'continuous_predictor').sum() == 14
profile_id_count_ok = (source_profile['role'] == 'identifier').sum() == 1
profile_target_count_ok = (source_profile['role'] == 'regression_target').sum() == 1
profile_roles_ok = (source_profile['role'] != 'unknown').all()
profile_dtype_ok = source_profile['observed_dtype'].notna().all()
profile_missing_ok = source_profile['missing_count'].notna().all()

print('source_profile has exactly 132 rows:', 'PASS' if profile_rows_ok else 'FAIL')
print('each source column appears exactly once:', 'PASS' if profile_unique_ok else 'FAIL')
print('116 categorical predictors represented:', 'PASS' if profile_cat_count_ok else 'FAIL')
print('14 continuous predictors represented:', 'PASS' if profile_cont_count_ok else 'FAIL')
print('1 identifier represented:', 'PASS' if profile_id_count_ok else 'FAIL')
print('1 regression target represented:', 'PASS' if profile_target_count_ok else 'FAIL')
print('every field has a role:', 'PASS' if profile_roles_ok else 'FAIL')
print('every field has dtype information:', 'PASS' if profile_dtype_ok else 'FAIL')
print('every field has missing-value information:', 'PASS' if profile_missing_ok else 'FAIL')

# PART 5 — SAVE MACHINE-READABLE PROFILE
artifacts_dir = '../artifacts'
if not os.path.exists(artifacts_dir):
    os.makedirs(artifacts_dir)

source_profile.to_csv(os.path.join(artifacts_dir, 'source_profile.csv'), index=False)
print('Source profile CSV saved to:', os.path.join(artifacts_dir, 'source_profile.csv'))

# PART 6 — FINAL TASK #3 SUMMARY
loss_numeric_summary = 'PASS' if loss_numeric else 'FAIL'
loss_finite_summary = 'PASS' if loss_no_infinite else 'FAIL'
loss_positive_summary = 'PASS' if loss_positive else 'FAIL'
cat_fields_validated = 'PASS' if len(categorical_profile) == 116 else 'FAIL'
cat_dtype_checks = [
    pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col]) or pd.api.types.is_string_dtype(df[col])
    for col in cat_cols
]
categorical_nominal = 'PASS' if all(cat_dtype_checks) else 'FAIL'
all_fields_profiled = 'PASS' if len(source_profile) == 132 else 'FAIL'
source_profile_generated = 'PASS' if os.path.exists(os.path.join(artifacts_dir, 'source_profile.csv')) else 'FAIL'

print('')
print('Gate 1 — Task #3: Field Validation and Source Profile')
print('Loss numeric:', loss_numeric_summary)
print('Loss finite:', loss_finite_summary)
print('Loss strictly positive:', loss_positive_summary)
print('116 categorical fields validated:', cat_fields_validated)
print('Categorical fields treated as nominal:', categorical_nominal)
print('All 132 fields profiled:', all_fields_profiled)
print('Source profile generated:', source_profile_generated)
print('Total fields profiled:', len(source_profile))
print('Number of categorical fields:', (source_profile['role'] == 'categorical_predictor').sum())
print('Minimum categorical cardinality:', categorical_profile['unique_count'].min())
print('Maximum categorical cardinality:', categorical_profile['unique_count'].max())
print('Top 5 highest-cardinality categorical fields:')
print(categorical_profile.sort_values('unique_count', ascending=False).head(5)[['column_name', 'unique_count']].to_string(index=False))
print('Loss minimum:', loss_min)
print('Loss maximum:', loss_max)
print('Number of fields containing missing values:', int((source_profile['missing_count'] > 0).sum()))

source_profile.head()

Loss dtype: float64
Loss unique count: 158223
Loss missing count: 0
Loss min: 0.67
Loss max: 121012.25
Loss non-finite count: 0
Loss values <= 0: 0
Loss numeric: PASS
Loss no missing values: PASS
Loss no infinite values: PASS
Loss strictly positive: PASS
Categorical profile rows: 116
Categorical profile sample:
Source profile rows: 132
source_profile has exactly 132 rows: PASS
each source column appears exactly once: PASS
116 categorical predictors represented: PASS
14 continuous predictors represented: PASS
1 identifier represented: PASS
1 regression target represented: PASS
every field has a role: PASS
every field has dtype information: PASS
every field has missing-value information: PASS
Source profile CSV saved to: ../artifacts/source_profile.csv

Gate 1 — Task #3: Field Validation and Source Profile
Loss numeric: PASS
Loss finite: PASS
Loss strictly positive: PASS
116 categorical fields validated: PASS
Categorical fields treated as nominal: PASS
All 132 fields profiled: PASS
Sourc

,column_name,role,observed_dtype,unique_count,missing_count,missing_percentage,observed_min,observed_max,expected_scale
0,id,identifier,int64,188318,0,0.0,1.0,587633.0,N/A
1,cat1,categorical_predictor,object,2,0,0.0,NaN,NaN,nominal categorical
2,cat2,categorical_predictor,object,2,0,0.0,NaN,NaN,nominal categorical
3,cat3,categorical_predictor,object,2,0,0.0,NaN,NaN,nominal categorical
4,cat4,categorical_predictor,object,2,0,0.0,NaN,NaN,nominal categorical


In [15]:
# Task 4 - Anomaly Register, Workflow Run and Verification
import datetime
import platform
import sys

# Programmatic Verification of Integrity Checks from Gate 1
audit_results = {}

# Check dataset
audit_results["file_size_bytes"] = os.path.getsize(
    "../data/allstate_claims_data.csv"
)
audit_results["row_count"] = len(df)
audit_results["column_count"] = len(df.columns)
audit_results["unique_id_count"] = df["id"].nunique()
audit_results["missing_cell_count"] = int(df.isna().sum().sum())
audit_results["duplicate_row_count"] = int(df.duplicated().sum())

# Check actual data values
audit_results["cont_in_bounds"] = bool(
    ((df[cont_cols] >= 0) & (df[cont_cols] <= 1)).all().all()
)
audit_results["loss_strictly_positive"] = bool((df["loss"] > 0).all()) # Loss must be positive
audit_results["artifact_profile_exists"] = os.path.exists(
    "../artifacts/source_profile.csv"
)

# Assertion checks
assert (
    audit_results["file_size_bytes"] == 70025339
), "FAIL: File size mismatch"
assert audit_results["row_count"] == 188318, "FAIL: Row count mismatch"
assert audit_results["column_count"] == 132, "FAIL: Column count mismatch"
assert (
    audit_results["unique_id_count"] == 188318
), "FAIL: Non-unique IDs detected"
assert (
    audit_results["missing_cell_count"] == 0
), "FAIL: Missing values detected"
assert audit_results["duplicate_row_count"] == 0, "FAIL: Duplicate rows detected"
assert audit_results[
    "cont_in_bounds"
], "FAIL: Continuous variables out of [0, 1]"
assert audit_results[
    "loss_strictly_positive"
], "FAIL: Non-positive loss values found"
assert audit_results[
    "artifact_profile_exists"
], "FAIL: Source profile artifact missing"

print("--- GATE 1 PROGRAMMATIC AUDIT: ALL CHECKS PASSED ---")
for check, status in audit_results.items():
  print(f"{check}: {status}")

# Workflow run metadata execution reciept
print("\n--- WORKFLOW RUN EXECUTION RECEIPT ---")
print("Execution Timestamp:", datetime.datetime.now().isoformat())
print("Python Version:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Pandas Version:", pd.__version__)
print("NumPy Version:", np.__version__)

--- GATE 1 PROGRAMMATIC AUDIT: ALL CHECKS PASSED ---
file_size_bytes: 70025339
row_count: 188318
column_count: 132
unique_id_count: 188318
missing_cell_count: 0
duplicate_row_count: 0
cont_in_bounds: True
loss_strictly_positive: True
artifact_profile_exists: True

--- WORKFLOW RUN EXECUTION RECEIPT ---
Execution Timestamp: 2026-09-13T13:58:21.281574
Python Version: 3.13.2
Platform: macOS-26.6.2-arm64-arm-64bit-Mach-O
Pandas Version: 2.3.2
NumPy Version: 2.3.3
